In [2]:
import re

# AST NODE
class Num:
    def __init__(self, value):
        self.value = value

class Var:
    def __init__(self, name):
        self.name = name

class BinOp:
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

# PARSER ERROR
class ParserError(Exception):
    pass


class MiniCompiler:
    def __init__(self, source, env):

        self._tokens = iter(
            re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[\^+*/()\-]', source)
            + ['?']
        )

        self._current = None
        self._env = env
        self._temp_count = 0

        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (
            expected == "ID" and self._current.isalnum()
        ):
            raise ParserError(f"Expected {expected}, found {self._current}")

        token = self._current
        self.advance()
        return token

    # FACTOR
    def factor(self):
        token = self._current

        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))

        elif token is not None and token.isalpha():
            self.advance()

            if token not in self._env:
                raise Exception(f"Variabel '{token}' tidak ditemukan")

            return Var(token)

        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node

        raise ParserError(f"Unexpected token: {token}")

    # POWER
    def power(self):
        node = self.factor()

        while self._current == '^':
            op = self._current
            self.advance()

            node = BinOp(node, op, self.factor())

        return node

    # TERM
    def term(self):
        node = self.power()

        while self._current in ('*', '/'):
            op = self._current
            self.advance()

            node = BinOp(node, op, self.power())

        return node

    # EXPR
    def expr(self):
        node = self.term()

        while self._current in ('+', '-'):
            op = self._current
            self.advance()

            node = BinOp(node, op, self.term())

        return node

    # GENERATE TAC
    def generate_tac(self, node):

        if isinstance(node, Num):
            return str(node.value)

        elif isinstance(node, Var):
            return node.name

        elif isinstance(node, BinOp):

            left = self.generate_tac(node.left)
            right = self.generate_tac(node.right)

            temp = f"t{self._temp_count}"
            self._temp_count += 1

            print(f"{temp} = {left} {node.op} {right}")

            return temp


# ====================================
# MAIN PROGRAM
# ====================================

# SYMBOL TABLE
symbol_table = {
    'a': 15,
    'b': 5,
    'c': 2
}

# SOURCE CODE
source_code = "a ^ 5 + b * c"

# BUAT COMPILER
compiler = MiniCompiler(source_code, symbol_table)

# PARSE AST
ast = compiler.expr()

# GENERATE TAC
print("=== THREE ADDRESS CODE ===")
compiler.generate_tac(ast)

=== THREE ADDRESS CODE ===
t0 = a ^ 5
t1 = b * c
t2 = t0 + t1


't2'